In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.inter_rater import fleiss_kappa
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [61]:
def load_annotations(file_path):
    """Load annotations from a JSON file."""
    # Open the JSON file in read mode with UTF-8 encoding
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)  # Parse and return the loaded JSON data


def parse_nlp_annotations(annotations1, annotations2):
    """Parse NLP annotations by matching based on text content and start position, and extracting POS tags."""
    
    # Create dictionaries for each annotation set based on annotation 'id' to quickly access annotations by ID
    annotations_dict1 = {annotation['id']: annotation for annotation in annotations1}
    annotations_dict2 = {annotation['id']: annotation for annotation in annotations2}
    # print(annotations_dict1)
    
    # Find common IDs between the two annotation sets to match the same text in both
    common_ids = set(annotations_dict1.keys()) & set(annotations_dict2.keys())

    pos_tags1, pos_tags2 = [], []  # Lists to store POS tags from both annotators

    # Iterate over common IDs to match annotations by text and start position
    for common_id in common_ids:
        labels1 = annotations_dict1[common_id]['label']
        labels2 = annotations_dict2[common_id]['label']

        # Compare each annotation in labels1 to labels2 for matching text and start position
        for label1 in labels1:
            text1 = label1['text'].strip()  # Clean text by removing extra spaces
            start1 = label1['start']
            for label2 in labels2:
                text2 = label2['text'].strip()  # Clean text for label2
                start2 = label2['start']
                
                # Match based on both text and start position
                if text1 == text2 and start1 == start2:
                    pos_tags1.append(label1['labels'][0])  # Store POS tag for annotator 1
                    pos_tags2.append(label2['labels'][0])  # Store POS tag for annotator 2
                    break  # No need to check further once a match is found
            else:
                # Print a warning when no match is found for a text-start pair
                print(f"Warning: No match for text '{text1}' with start position {start1} in ID {common_id}. Skipping.")
    
    return pos_tags1, pos_tags2  # Return the matched POS tags


def calculate_cohen_kappa(pos_tags1, pos_tags2):
    """Calculate Cohen's Kappa score and interpret the agreement between two annotators."""
    
    # Get the set of all unique labels across both annotators to understand the full set of POS tags
    all_labels = set(pos_tags1 + pos_tags2)
    print("Number of unique labels:", len(all_labels))
    print("Labels used:", all_labels)

    # Identify mismatches between annotators by comparing POS tags
    mismatches = [(i, tag1, tag2) for i, (tag1, tag2) in enumerate(zip(pos_tags1, pos_tags2)) if tag1 != tag2]
    
    if mismatches:
        # Print mismatches if any are found
        print("\nMismatched Annotations (Annotators Disagree):")
        for mismatch in mismatches:
            print(f"Index: {mismatch[0]}, Annotator 1: {mismatch[1]}, Annotator 2: {mismatch[2]}")
    else:
        # If no mismatches, print that all annotators agree
        print("\nNo mismatches detected. All annotators agree.")

    # Compute Cohen's Kappa score using sklearn's function
    
    kappa_score = cohen_kappa_score(pos_tags1, pos_tags2)
    print("Cohen's Kappa:", kappa_score)

    # Interpret the Cohen's Kappa score based on its value
    if kappa_score < 0:
        print("No agreement")
    elif kappa_score < 0.2:
        print("Slight agreement")
    elif kappa_score < 0.4:
        print("Fair agreement")
    elif kappa_score < 0.6:
        print("Moderate agreement")
    elif kappa_score < 0.8:
        print("Substantial agreement")
    else:
        print("Almost perfect agreement")

    return kappa_score  # Return the computed Kappa score

# -----------------------------------------------------------------------------------------------------------------------------------------------------------

def generate_confusion_matrix(pos_tags1, pos_tags2):
    """Generate a confusion matrix comparing two annotators' POS tags."""
    
    # Step 1: Identify unique labels
    all_labels = sorted(set(pos_tags1 + pos_tags2))  # Ensure order consistency
    label_to_index = {label: i for i, label in enumerate(all_labels)}
    
    # Step 2: Initialize empty N × N matrix
    N = len(all_labels)
    confusion_matrix = np.zeros((N, N), dtype=int)
    
    # Step 3: Fill the confusion matrix
    for tag1, tag2 in zip(pos_tags1, pos_tags2):
        i, j = label_to_index[tag1], label_to_index[tag2]
        confusion_matrix[i, j] += 1
    
    # Step 4: Convert to a DataFrame for better readability
    df_matrix = pd.DataFrame(confusion_matrix, index=all_labels, columns=all_labels)
    
    # Step 5: Print the matrix in a formatted way
    print("\nConfusion Matrix:")

    return df_matrix

def compute_cohen_kappa(conf_matrix):
    """Compute Cohen's Kappa score from a confusion matrix."""
    total = np.sum(conf_matrix)
    p_o = np.trace(conf_matrix) / total  # Observed agreement
    
    row_sums = np.sum(conf_matrix, axis=1)
    col_sums = np.sum(conf_matrix, axis=0)
    p_e = np.sum((row_sums * col_sums) / (total ** 2))  # Expected agreement
    
    kappa = (p_o - p_e) / (1 - p_e)
    return kappa



# @@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@


def parse_cv_annotations(file_data):
    """Parse CV annotations and extract image name and label."""
    extracted_data = {}
    
    # Loop over the data to extract image names and their corresponding labels
    for item in file_data:
        # Split the 'image' field to extract just the image name (e.g., 'img_1.jpg')
        image_name = item['image'].split('-')[-1]
        label = item['choice']  # The label assigned to the image
        extracted_data[image_name] = label  # Store the label for the image
    
    return extracted_data  # Return the parsed data


def combine_annotations(data1_parsed, data2_parsed, data3_parsed):
    """Combine annotations from multiple sources based on image name."""
    combined_data = {}
    
    # Combine data from all sources by taking the union of all unique image names
    for image_name in set(data1_parsed.keys()).union(data2_parsed.keys()).union(data3_parsed.keys()):
        combined_data[image_name] = []
        
        # Append the label from each data source if the image name exists
        if image_name in data1_parsed:
            combined_data[image_name].append(data1_parsed[image_name])
        if image_name in data2_parsed:
            combined_data[image_name].append(data2_parsed[image_name])
        if image_name in data3_parsed:
            combined_data[image_name].append(data3_parsed[image_name])
    
    return combined_data  # Return the combined data


def calculate_fleiss_kappa(combined_data):
    """Calculate Fleiss Kappa score and interpret the agreement for multiple annotators."""
    # Convert combined data into a DataFrame
    df = pd.DataFrame.from_dict(combined_data, orient='index')
  
    
    # Identify mismatches where annotators disagree
    mismatches = df[df.nunique(axis=1) > 1]
    all_labels = set(df.values.flatten())  # Get the set of all unique labels used
    
    print("Number of unique labels:", len(all_labels))
    print("Labels used:", all_labels)

    if not mismatches.empty:
        print("\nMismatched Annotations (Annotators Disagree):")
        print(mismatches)
    else:
        print("\nNo mismatches detected. All annotators agree.")

    # Map the labels to numerical values for the Fleiss Kappa calculation
    label_map = {label: i for i, label in enumerate(all_labels)}
    df_mapped = df.applymap(lambda x: label_map[x])  # Apply the mapping to the DataFrame
    
    # Create a rating table with counts of each label per image
    rating_table = []
    for row in df_mapped.itertuples(index=False):
        counts = [0] * len(label_map)  # Initialize count list for labels
        for label in row:
            counts[label] += 1  # Count occurrences of each label
        rating_table.append(counts)  # Add the count list to the rating table
    print("\nRating Table:")
    print(rating_table)


    
    # Calculate Fleiss Kappa score
    fleiss_kappa_score = fleiss_kappa(rating_table)
    print("\nFleiss Kappa Score:", fleiss_kappa_score)
    
    # Interpret the Fleiss Kappa score
    if 0.80 <= fleiss_kappa_score <= 1.00:
        print("Very good agreement")
    elif 0.60 <= fleiss_kappa_score < 0.80:
        print("Good agreement")
    elif 0.40 <= fleiss_kappa_score < 0.60:
        print("Moderate agreement")
    elif 0.20 <= fleiss_kappa_score < 0.40:
        print("Fair agreement")
    else:
        print("Poor agreement")

    return fleiss_kappa_score  # Return the computed Fleiss Kappa score


# -----------------------------------------------------------------------------------------------------------------------------------------------------------

def fleiss_kappa(rating_table):
    """
    Compute Fleiss' Kappa for multiple annotators.
    rating_table: List of lists where each row contains the count of ratings per category.
    """
    rating_table = np.array(rating_table)
    N, k = rating_table.shape  # N = number of subjects, k = categories
    
    n = np.sum(rating_table[0])  # Total annotators per subject (assumed constant)

    # Compute P_i for each row
    P_i = np.sum((rating_table * (rating_table - 1)), axis=1) / (n * (n - 1))
    
    # Compute P (observed agreement)
    P = np.mean(P_i)

    # Compute category probabilities p_j
    p_j = np.sum(rating_table, axis=0) / (N * n)

    # Compute P_e (expected agreement)
    P_e = np.sum(p_j ** 2)

    # Compute Fleiss' Kappa
    kappa = (P - P_e) / (1 - P_e)

    return kappa



def summary(kappa_score, fleiss_kappa_score):
    """Print a summary of Cohen's Kappa and Fleiss Kappa scores."""
    print("\nSummary:")
    print(f"Cohen's Kappa Score: {kappa_score}")
    print(f"Fleiss Kappa Score: {fleiss_kappa_score}")


In [62]:

# Main execution starts here
annotations1 = load_annotations('NLP_23110065.json')
annotations2 = load_annotations('NLP_23110066.json')
pos_tags1, pos_tags2 = parse_nlp_annotations(annotations1, annotations2)
kappa_score = calculate_cohen_kappa(pos_tags1, pos_tags2)
print("-" * 130)

data1 = load_annotations('CV_23110065.json')
data2 = load_annotations('CV_23110066.json')
data3 = load_annotations('CV_third-member.json')
data1_parsed = parse_cv_annotations(data1)
data2_parsed = parse_cv_annotations(data2)
data3_parsed = parse_cv_annotations(data3)
combined_data = combine_annotations(data1_parsed, data2_parsed, data3_parsed)
fleiss_kappa_score = calculate_fleiss_kappa(combined_data)

summary(kappa_score, fleiss_kappa_score)


Number of unique labels: 14
Labels used: {'NUM', 'NOUN', 'VERB', 'X', 'PRON_WH', 'ADJ', 'ADP', 'PROPN', 'ADV', 'PRON', 'PART_NEG', 'DET', 'PART', 'CONJ'}

Mismatched Annotations (Annotators Disagree):
Index: 49, Annotator 1: ADV, Annotator 2: PART
Index: 74, Annotator 1: NOUN, Annotator 2: PROPN
Index: 82, Annotator 1: NUM, Annotator 2: NOUN
Index: 88, Annotator 1: CONJ, Annotator 2: ADP
Index: 90, Annotator 1: NUM, Annotator 2: NOUN
Index: 91, Annotator 1: PRON_WH, Annotator 2: NOUN
Index: 96, Annotator 1: PROPN, Annotator 2: NOUN
Index: 110, Annotator 1: PART, Annotator 2: ADP
Index: 112, Annotator 1: ADJ, Annotator 2: PART
Index: 116, Annotator 1: NOUN, Annotator 2: PROPN
Index: 176, Annotator 1: ADV, Annotator 2: ADP
Index: 185, Annotator 1: NOUN, Annotator 2: ADJ
Index: 259, Annotator 1: ADP, Annotator 2: ADJ
Index: 269, Annotator 1: PRON, Annotator 2: ADJ
Index: 271, Annotator 1: PRON, Annotator 2: PROPN
Index: 303, Annotator 1: NOUN, Annotator 2: PROPN
Index: 308, Annotator 1: V

### Output Interpretation

#### **Cohen's Kappa Score: 0.9273 (Almost perfect agreement)**
- **Significance**: The Cohen's Kappa score of **0.9273** indicates that the agreement between the two annotators is *almost perfect*. This score is very close to the maximum possible value of **1.0**, reflecting a high level of consistency between the annotators' POS tag assignments. The formula for Cohen's Kappa is:

$$
\kappa = \frac{P_o - P_e}{1 - P_e}
$$

Where:
- \( P_o \) is the observed agreement (i.e., the proportion of times both annotators agree),
- \( P_e \) is the expected agreement by chance.

#### **Mismatched Annotations**
- A number of mismatched annotations are detected, where Annotator 1 and Annotator 2 disagree on the POS tag for certain words (e.g., **ADV** vs **PART**, **NOUN** vs **PROPN**). 
  - **Example**: At index 49, Annotator 1 labeled the word as **ADV** (adverb), while Annotator 2 labeled it as **PART** (particle).

#### **Fleiss Kappa Score: 0.8667 (Very good agreement)**
- **Significance**: The Fleiss Kappa score of **0.8667** suggests that the three annotators (in this case) show a **very good agreement** with respect to the labels assigned. A score above **0.80** indicates strong consensus and reflects that most of the images were consistently labeled, with very few discrepancies. The formula for Fleiss' Kappa is:

$$
\kappa_f = \frac{P_o - P_e}{1 - P_e}
$$

Where:
- \( P_o \) is the observed agreement (the proportion of times annotators agree across all items),
- \( P_e \) is the expected agreement by chance for multiple annotators.

---

Some words are skipped because of the following reasons:
- One annotator might have assigned a POS tag for a word, while the other may have missed it or assigned a different tag.
- One annotator might have assigned a POS tag for a word, while the other may have grouped that word with nearby words (i.e., different tokenization).

The skipped words do not significantly affect the overall agreement score because only common annotations are being compared.

---

### Fleiss' Kappa Formula: \( P_o \) and \( P_e \)

#### **Observed Agreement \( P_o \)**

The observed agreement \( P_o \) is calculated as the average extent to which raters agree on the elements. It is given by:

$$
P_o = \frac{1}{Nn(n-1)} \left[ \sum_{i=1}^{N} \sum_{j=1}^{k} n_{ij}^2 - Nn \right]
$$

Where:
- \( N \) is the total number of elements
- \( n \) is the number of ratings per element
- \( k \) is the number of categories
- \( n_{ij} \) represents the number of raters who assigned the \( i \)-th element to the \( j \)-th category

#### **Expected Agreement \( P_e \)**

The expected agreement \( P_e \) is the sum of the squared proportions for each category. It is given by:

$$
P_e = \sum_{j=1}^{k} p_j^2
$$

Where:
- \( p_j \) is the proportion of assignments to category \( j \), calculated as:

$$
p_j = \frac{1}{Nn} \sum_{i=1}^{N} n_{ij}
$$


In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

def generate_rating_table(pos_tags_list):
    """Generate a rating table for multiple annotators' POS tags."""
    num_annotators = len(pos_tags_list)
    num_samples = len(pos_tags_list[0])
    
    # Identify all unique labels
    all_labels = sorted(set(tag for tags in pos_tags_list for tag in tags))
    label_to_index = {label: i for i, label in enumerate(all_labels)}
    
    # Initialize rating table (num_samples x num_labels)
    rating_table = np.zeros((num_samples, len(all_labels)), dtype=int)
    
    # Populate the rating table
    for i in range(num_samples):
        counts = Counter(pos_tags_list[j][i] for j in range(num_annotators))
        for label, count in counts.items():
            rating_table[i, label_to_index[label]] = count
    
    return pd.DataFrame(rating_table, columns=all_labels)

def compute_fleiss_kappa(rating_table):
    """Compute Fleiss' Kappa from a rating table."""
    n = np.sum(rating_table, axis=1)[0]  # Number of annotators per sample
    N, k = rating_table.shape  # N = number of samples, k = number of labels
    
    # Compute p_i (proportion agreement per sample)
    P_i = (np.sum(rating_table**2, axis=1) - n) / (n * (n - 1))
    P_bar = np.mean(P_i)
    
    # Compute p_e (expected agreement)
    p_e = np.sum((np.sum(rating_table, axis=0) / (N * n))**2)
    
    # Compute Fleiss' Kappa
    kappa = (P_bar - p_e) / (1 - p_e)
    return kappa

def parse_nlp_annotations(annotations1, annotations2, annotations3):
    """Parse NLP annotations from three annotators, matching based on text content and start position."""
    annotations_dict1 = {annotation['id']: annotation for annotation in annotations1}
    annotations_dict2 = {annotation['id']: annotation for annotation in annotations2}
    annotations_dict3 = {annotation['id']: annotation for annotation in annotations3}
    
    common_ids = set(annotations_dict1.keys()) & set(annotations_dict2.keys()) & set(annotations_dict3.keys())
    
    pos_tags1, pos_tags2, pos_tags3 = [], [], []
    
    for common_id in common_ids:
        labels1 = annotations_dict1[common_id]['label']
        labels2 = annotations_dict2[common_id]['label']
        labels3 = annotations_dict3[common_id]['label']
        
        for label1 in labels1:
            text1 = label1['text'].strip()
            start1 = label1['start']
            for label2 in labels2:
                text2 = label2['text'].strip()
                start2 = label2['start']
                for label3 in labels3:
                    text3 = label3['text'].strip()
                    start3 = label3['start']
                    
                    if text1 == text2 == text3 and start1 == start2 == start3:
                        pos_tags1.append(label1['labels'][0])
                        pos_tags2.append(label2['labels'][0])
                        pos_tags3.append(label3['labels'][0])
                        break
                else:
                    print(f"Warning: No match for text '{text1}' with start position {start1} in ID {common_id}. Skipping.")
    
    return pos_tags1, pos_tags2, pos_tags3